### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_transaction_value",
    dataset_year="2018",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/santander-value-prediction-challenge",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-value-prediction-challenge -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_transaction_value && mv train.csv local-data-warehouse/santander_transaction_value/
""",
    # References
    academic_reference_bibtex=r"""@misc{McDonald2018SantanderValuePredictionChallenge,
  author = {Mark McDonald and Mercedes Piedra and Sohier Dane and Soraya Jimenez},
  title  = {Santander Value Prediction Challenge},
  year   = {2018},
  howpublished = {\url{https://kaggle.com/competitions/santander-value-prediction-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="McDonald2018SantanderValuePredictionChallenge",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized.
- We follow the preprocessing by Tschalzev et al. (https://arxiv.org/abs/2407.02112), which follows https://www.kaggle.com/competitions/santander-value-prediction-challenge/discussion/63919
- Since we restrict to the train.csv, we do not exploit the test leak of the competition.
- The expert preprocessing aggregates over feature groups per row to create the new features for the final dataset. Feature groups for aggregation are found by searching for chains of features (?).
- The data contains a very small number of duplicates. We remove them to avoid leakage in our own splits.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="regression",
    objective_metric_name="rmsle",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

og_df = pd.read_csv(dataset_mold.path / "train.csv")
# print("Loaded data shape:", df.shape)

In [ ]:
train = og_df.copy()

# Code adapted from https://github.com/atschalz/dc_tabeval/blob/main/datasets.py#L2120 w/o test leak part
import gc
features = [
    'f190486d6', '58e2e02e6', 'eeb9cd3aa', '9fd594eec', '6eef030c1',
    '15ace8c9f', 'fb0f5dbfe', '58e056e12', '20aa07010', '024c577b9',
    'd6bb78916', 'b43a7cfd5', '58232a6fb', '1702b5bf0', '324921c7b',
    '62e59a501', '2ec5b290f', '241f0f867', 'fb49e4212', '66ace2992',
    'f74e8f13d', '5c6487af1', '963a49cdc', '26fc93eb7', '1931ccfdd',
    '703885424', '70feb1494', '491b9ee45', '23310aa6f', 'e176a204a',
    '6619d81fc', '1db387535', 'fc99f9426', '91f701ba2', '0572565c2',
    '190db8488', 'adb64ff71', 'c47340d97', 'c5a231d81', '0ff32eb98'
]
extra_features = []

train = train
train_t = train.drop(['target'], axis = 1, inplace=False)
train_t.set_index('ID', inplace=True)
train_t = train_t.T

#run this iteratively until you have no more links. Then prune
def chain_pairs(ordered_items):
    ordered_chains = []
    links_found = 0
    for i_1, op_chain in enumerate(ordered_items.copy()[:]):
        if op_chain[0] != op_chain[1]:
            end_chain = op_chain[-1]
            for i_2, op in enumerate(ordered_items.copy()[:]):
                if (end_chain == op[0]):
                    links_found += 1
                    op_chain.extend(op[1:])
                    end_chain = op_chain[-1]

            ordered_chains.append(op_chain)
    return links_found, ordered_chains

def prune_chain(ordered_chain):

    ordered_chain = sorted(ordered_chain, key=len, reverse=True)
    new_chain = []
    id_lookup = {}
    for oc in ordered_chain:
        id_already_in_chain = False
        for idd in oc:
            if idd in id_lookup:
                id_already_in_chain = True
            id_lookup[idd] = idd

        if not id_already_in_chain:
            new_chain.append(oc)
    return sorted(new_chain, key=len, reverse=True)

def find_new_ordered_features(ordered_ids, data_t):
    data = data_t.copy()

    f1 = ordered_ids[0][:-1]
    f2 = ordered_ids[0][1:]
    for ef in ordered_ids[1:]:
        f1 += ef[:-1]
        f2 += ef[1:]

    d1 = data[f1].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d1['ID'] = data.index
    gc.collect()
    d2 = data[f2].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d2['ID'] = data.index
    gc.collect()
    d3 = d2[~d2.duplicated(['key'], keep=False)]
    d4 = d1[~d1.duplicated(['key'], keep=False)]
    d5 = d4.merge(d3, how='inner', on='key')

    d_feat = d1.merge(d5, how='left', on='key')
    d_feat.fillna(0, inplace=True)

    ordered_features = list(d_feat[['ID_x', 'ID_y']][d_feat.ID_x != 0].apply(list, axis=1))
    del d1,d2,d3,d4,d5,d_feat
    gc.collect()

    links_found = 1
    while links_found > 0:
        links_found, ordered_features = chain_pairs(ordered_features)

    ordered_features = prune_chain(ordered_features)
    #make lookup of all features found so far
    found = {}
    for ef in extra_features:
        found[ef[0]] = ef
    found [features[0]] = features

    new_feature_sets = []
    for of in ordered_features:
        if len(of) >= 40:
            if of[0] not in found:
                new_feature_sets.append(of)

    return new_feature_sets

def add_new_feature_sets(data, data_t):

    # print ('\nData Shape:', data.shape)
    f1 = features[:-1]
    f2 = features[1:]

    for ef in extra_features:
        f1 += ef[:-1]
        f2 += ef[1:]

    d1 = data[f1].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d1['ID'] = data['ID']
    gc.collect()
    d2 = data[f2].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d2['ID'] = data['ID']
    gc.collect()
    #print('here')
    d3 = d2[~d2.duplicated(['key'], keep=False)]
    del d2
    d4 = d1[~d1.duplicated(['key'], keep=False)]
    #print('here')
    d5 = d4.merge(d3, how='inner', on='key')
    del d4
    d = d1.merge(d5, how='left', on='key')
    d.fillna(0, inplace=True)
    #print('here')
    ordered_ids = list(d[['ID_x', 'ID_y']][d.ID_x != 0].apply(list, axis=1))
    del d1,d3,d5,d
    gc.collect()

    links_found = 1
    while links_found > 0:
        links_found, ordered_ids = chain_pairs(ordered_ids)
        #print(links_found)

    # print ('OrderedIds:', len(ordered_ids))
    #Make distinct ordered id chains
    ordered_ids = prune_chain(ordered_ids)
    # print ('OrderedIds Pruned:', len(ordered_ids))

    #look for ordered features with new ordered id chains
    new_feature_sets = find_new_ordered_features(ordered_ids, data_t)

    extra_features.extend(new_feature_sets)
    # print('New Feature Count:', len(new_feature_sets))
    # print('Extra Feature Count:', len(extra_features))

add_new_feature_sets(train,train_t)
add_new_feature_sets(train,train_t)
add_new_feature_sets(train,train_t)

del train_t
gc.collect()

### Create 40 features
extra_features_list = []

for ef in extra_features:
    extra_features_list.extend(ef)

extra_features_list.extend(features)

#This makes the 100 40 length feature groups into 40 100 length feature groups.
feats = pd.DataFrame(extra_features)
time_features = []
for c in feats.columns[:]:
    time_features.append([f for f in feats[c].values if f is not None])


#Make a bunch of different feature groups to build aggregates from
agg_features = []
all_cols = train.columns.drop(['ID', 'target'])
agg_features.append(all_cols)
agg_features.append([c for c in all_cols if c not in extra_features_list])
agg_features.append(extra_features_list)
agg_features.extend(time_features)
agg_features.extend(extra_features)

def add_new_features(source, dest, feats):
    high = source[feats].max(axis=1)
    high.name = 'high_{}_{}'.format(feats[0], len(feats))
    mean = source[feats].replace(0, np.nan).mean(axis=1)
    mean.name = 'mean_{}_{}'.format(feats[0], len(feats))
    low = source[feats].replace(0, np.nan).min(axis=1)
    low.name = 'low_{}_{}'.format(feats[0], len(feats))
    median = source[feats].replace(0, np.nan).median(axis=1)
    median.name = 'median_{}_{}'.format(feats[0], len(feats))
    sum = source[feats].sum(axis=1)
    sum.name = 'sum_{}_{}'.format(feats[0], len(feats))
    stddev = source[feats].std(axis=1)
    stddev.name = 'stddev_{}_{}'.format(feats[0], len(feats))
    first_nonZero= np.log1p(source[feats].replace(0, np.nan).bfill(axis=1).iloc[:, 0])
    first_nonZero.name = 'first_nonZero_{}_{}'.format(feats[0], len(feats))
    last_nonZero = np.log1p(source[feats[::-1]].replace(0, np.nan).bfill(axis=1).iloc[:, 0])
    last_nonZero.name = 'last_nonZero_{}_{}'.format(feats[0], len(feats))
    nb_nans =  source[feats].replace(0, np.nan).isnull().sum(axis=1)
    nb_nans.name = 'nb_nans_{}_{}'.format(feats[0], len(feats))
    unique = source[feats].nunique(axis=1)
    unique.name = 'unique_{}_{}'.format(feats[0], len(feats))

    dest = pd.concat([dest, high, mean, low, median, sum, stddev, first_nonZero, last_nonZero, nb_nans, unique],axis=1)
    return dest


train_feats = pd.DataFrame()
for i, ef in list(enumerate(agg_features)):
    train_feats = add_new_features(train, train_feats, ef)


df = train_feats
df[task_mold.target_column_name] = np.log1p(train.target.values)

# Remove duplicates
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name])
df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()